In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/home-credit-default-risk/sample_submission.csv
/kaggle/input/competitions/home-credit-default-risk/bureau_balance.csv
/kaggle/input/competitions/home-credit-default-risk/POS_CASH_balance.csv
/kaggle/input/competitions/home-credit-default-risk/application_train.csv
/kaggle/input/competitions/home-credit-default-risk/HomeCredit_columns_description.csv
/kaggle/input/competitions/home-credit-default-risk/application_test.csv
/kaggle/input/competitions/home-credit-default-risk/previous_application.csv
/kaggle/input/competitions/home-credit-default-risk/credit_card_balance.csv
/kaggle/input/competitions/home-credit-default-risk/installments_payments.csv
/kaggle/input/competitions/home-credit-default-risk/bureau.csv


In [2]:
import pandas as pd

df = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/application_train.csv')
print(df.shape)
print(df.isnull().mean().sort_values(ascending=False).head(20))
print(df['TARGET'].value_counts(normalize=True))

(307511, 122)
COMMONAREA_AVG              0.698723
COMMONAREA_MODE             0.698723
COMMONAREA_MEDI             0.698723
NONLIVINGAPARTMENTS_MEDI    0.694330
NONLIVINGAPARTMENTS_MODE    0.694330
NONLIVINGAPARTMENTS_AVG     0.694330
FONDKAPREMONT_MODE          0.683862
LIVINGAPARTMENTS_AVG        0.683550
LIVINGAPARTMENTS_MEDI       0.683550
LIVINGAPARTMENTS_MODE       0.683550
FLOORSMIN_MODE              0.678486
FLOORSMIN_AVG               0.678486
FLOORSMIN_MEDI              0.678486
YEARS_BUILD_AVG             0.664978
YEARS_BUILD_MODE            0.664978
YEARS_BUILD_MEDI            0.664978
OWN_CAR_AGE                 0.659908
LANDAREA_MEDI               0.593767
LANDAREA_AVG                0.593767
LANDAREA_MODE               0.593767
dtype: float64
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


In [3]:
col_desc = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/HomeCredit_columns_description.csv', encoding='latin1')
col_desc.head(20)

,Unnamed: 0,Table,Row,Description,Special
0,1,application_{train|test}.csv,SK_ID_CURR,ID of loan in our sample,NaN
1,2,application_{train|test}.csv,TARGET,Target variable (1 - client with payment diffi...,NaN
2,5,application_{train|test}.csv,NAME_CONTRACT_TYPE,Identification if loan is cash or revolving,NaN
3,6,application_{train|test}.csv,CODE_GENDER,Gender of the client,NaN
4,7,application_{train|test}.csv,FLAG_OWN_CAR,Flag if the client owns a car,NaN
5,8,application_{train|test}.csv,FLAG_OWN_REALTY,Flag if client owns a house or flat,NaN
6,9,application_{train|test}.csv,CNT_CHILDREN,Number of children the client has,NaN
7,10,application_{train|test}.csv,AMT_INCOME_TOTAL,Income of the client,NaN
8,11,application_{train|test}.csv,AMT_CREDIT,Credit amount of the loan,NaN
9,12,application_{train|test}.csv,AMT_ANNUITY,Loan annuity,NaN


In [4]:
missing_pct = df.isnull().mean().sort_values(ascending=False)
cols_to_drop = missing_pct[missing_pct > 0.40].index.tolist()
print(f"Total columns to drop: {len(cols_to_drop)}")
print(cols_to_drop)

df_clean = df.drop(columns=cols_to_drop)
print(f"New shape: {df_clean.shape}")

Total columns to drop: 49
['COMMONAREA_AVG', 'COMMONAREA_MODE', 'COMMONAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAPARTMENTS_AVG', 'FONDKAPREMONT_MODE', 'LIVINGAPARTMENTS_AVG', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAPARTMENTS_MODE', 'FLOORSMIN_MODE', 'FLOORSMIN_AVG', 'FLOORSMIN_MEDI', 'YEARS_BUILD_AVG', 'YEARS_BUILD_MODE', 'YEARS_BUILD_MEDI', 'OWN_CAR_AGE', 'LANDAREA_MEDI', 'LANDAREA_AVG', 'LANDAREA_MODE', 'BASEMENTAREA_MODE', 'BASEMENTAREA_MEDI', 'BASEMENTAREA_AVG', 'EXT_SOURCE_1', 'NONLIVINGAREA_MODE', 'NONLIVINGAREA_AVG', 'NONLIVINGAREA_MEDI', 'ELEVATORS_AVG', 'ELEVATORS_MEDI', 'ELEVATORS_MODE', 'WALLSMATERIAL_MODE', 'APARTMENTS_AVG', 'APARTMENTS_MEDI', 'APARTMENTS_MODE', 'ENTRANCES_MODE', 'ENTRANCES_MEDI', 'ENTRANCES_AVG', 'LIVINGAREA_AVG', 'LIVINGAREA_MEDI', 'LIVINGAREA_MODE', 'HOUSETYPE_MODE', 'FLOORSMAX_MODE', 'FLOORSMAX_AVG', 'FLOORSMAX_MEDI', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BEGINEXPLUATATION_AVG', 'TOTALAREA_M

In [5]:
remaining_missing = df_clean.isnull().mean().sort_values(ascending=False)
print(remaining_missing[remaining_missing > 0])

OCCUPATION_TYPE               0.313455
EXT_SOURCE_3                  0.198253
AMT_REQ_CREDIT_BUREAU_YEAR    0.135016
AMT_REQ_CREDIT_BUREAU_QRT     0.135016
AMT_REQ_CREDIT_BUREAU_HOUR    0.135016
AMT_REQ_CREDIT_BUREAU_MON     0.135016
AMT_REQ_CREDIT_BUREAU_WEEK    0.135016
AMT_REQ_CREDIT_BUREAU_DAY     0.135016
NAME_TYPE_SUITE               0.004201
DEF_30_CNT_SOCIAL_CIRCLE      0.003320
OBS_30_CNT_SOCIAL_CIRCLE      0.003320
OBS_60_CNT_SOCIAL_CIRCLE      0.003320
DEF_60_CNT_SOCIAL_CIRCLE      0.003320
EXT_SOURCE_2                  0.002146
AMT_GOODS_PRICE               0.000904
AMT_ANNUITY                   0.000039
CNT_FAM_MEMBERS               0.000007
DAYS_LAST_PHONE_CHANGE        0.000003
dtype: float64


In [7]:
print(df_clean.dtypes.value_counts())
categorical_cols = df_clean.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns: {len(categorical_cols)}")
print(categorical_cols)

int64      41
float64    20
object     12
Name: count, dtype: int64
Categorical columns: 12
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE']


In [8]:
# Categorical columns - mode imputation
categorical_cols = df_clean.select_dtypes(include='object').columns.tolist()
for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# Numerical columns - median imputation
numerical_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_cols.remove('TARGET')  # target column ko touch nahi karna
numerical_cols.remove('SK_ID_CURR')  # ID column bhi nahi

for col in numerical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Confirm no missing values left
print(f"Total missing values remaining: {df_clean.isnull().sum().sum()}")

Total missing values remaining: 0


In [9]:
df_model = df_clean.drop(columns=['SK_ID_CURR'])
print(df_model.shape)

(307511, 72)


In [10]:
# One-hot encode categorical columns (for the baseline ANN)
df_encoded = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)
print(f"Shape after one-hot encoding: {df_encoded.shape}")

Shape after one-hot encoding: (307511, 172)


In [11]:
X = df_encoded.drop(columns=['TARGET'])
y = df_encoded['TARGET']
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (307511, 171)
y shape: (307511,)


In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Scaled X shape: {X_scaled.shape}")
print(f"Mean of first column (should be ~0): {X_scaled[:, 0].mean():.4f}")
print(f"Std of first column (should be ~1): {X_scaled[:, 0].std():.4f}")

Scaled X shape: (307511, 171)
Mean of first column (should be ~0): -0.0000
Std of first column (should be ~1): 1.0000


In [15]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Just to confirm the split works correctly
for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, y)):
    print(f"Fold {fold+1}: Train size = {len(train_idx)}, Val size = {len(val_idx)}")
    print(f"  Train target distribution: {y.iloc[train_idx].value_counts(normalize=True).to_dict()}")
    print(f"  Val target distribution: {y.iloc[val_idx].value_counts(normalize=True).to_dict()}")

Fold 1: Train size = 246008, Val size = 61503
  Train target distribution: {0: 0.9192709180189262, 1: 0.08072908198107379}
  Val target distribution: {0: 0.9192722306228964, 1: 0.08072776937710356}
Fold 2: Train size = 246009, Val size = 61502
  Train target distribution: {0: 0.9192712461739204, 1: 0.08072875382607954}
  Val target distribution: {0: 0.9192709180189262, 1: 0.08072908198107379}
Fold 3: Train size = 246009, Val size = 61502
  Train target distribution: {0: 0.9192712461739204, 1: 0.08072875382607954}
  Val target distribution: {0: 0.9192709180189262, 1: 0.08072908198107379}
Fold 4: Train size = 246009, Val size = 61502
  Train target distribution: {0: 0.9192712461739204, 1: 0.08072875382607954}
  Val target distribution: {0: 0.9192709180189262, 1: 0.08072908198107379}
Fold 5: Train size = 246009, Val size = 61502
  Train target distribution: {0: 0.9192712461739204, 1: 0.08072875382607954}
  Val target distribution: {0: 0.9192709180189262, 1: 0.08072908198107379}


In [17]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)
print(f"Class weights: {class_weights}")
# class_weights[0] = weight for class 0 (repaid)
# class_weights[1] = weight for class 1 (default) -- yeh zyada hoga

Class weights: [0.54390914 6.19357503]


In [18]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape}, Val: {X_val.shape}")

Train: (246008, 171), Val: (61503, 171)


In [20]:
import torch
import torch.nn as nn

# Convert to PyTorch tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train.values).reshape(-1, 1)
X_val_t = torch.FloatTensor(X_val)
y_val_t = torch.FloatTensor(y_val.values).reshape(-1, 1)

print(f"X_train_t: {X_train_t.shape}, y_train_t: {y_train_t.shape}")

# GPU available hai to use karo (Kaggle Notebook settings mein GPU enable karna zaroori)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

X_train_t, y_train_t = X_train_t.to(device), y_train_t.to(device)
X_val_t, y_val_t = X_val_t.to(device), y_val_t.to(device)

X_train_t: torch.Size([246008, 171]), y_train_t: torch.Size([246008, 1])
Using device: cuda


In [23]:
class LoanANN(nn.Module):
    def __init__(self, input_size):
        super(LoanANN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.25),
            
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.2),
            
            nn.Linear(32, 1)  # Sigmoid alag se lagayenge loss function mein (numerically stable)
        )
    
    def forward(self, x):
        return self.model(x)

input_size = X_train.shape[1]  # 171
model = LoanANN(input_size).to(device)
print(model)

LoanANN(
  (model): Sequential(
    (0): Linear(in_features=171, out_features=128, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): Dropout(p=0.25, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): ReLU()
    (10): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [24]:
import torch.optim as optim

# pos_weight = weight for positive class (class 1 = default)
# BCEWithLogitsLoss khud sigmoid apply karta hai internally (isliye model mein sigmoid nahi lagaya)
pos_weight = torch.tensor([class_weights[1] / class_weights[0]]).to(device)
print(f"pos_weight: {pos_weight}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

pos_weight: tensor([11.3872], device='cuda:0', dtype=torch.float64)


In [25]:
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, TensorDataset

# Batching zaroori hai 246K rows ke liye (poora data ek saath GPU mein nahi daal sakte)
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)

n_epochs = 30
best_f1 = 0
patience_counter = 0
early_stop_patience = 5

for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_train_loss = total_loss / len(train_loader)
    
    # Validation
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val_t)
        val_loss = criterion(val_outputs, y_val_t).item()
        val_preds = (torch.sigmoid(val_outputs) >= 0.5).float()
        val_f1 = f1_score(y_val_t.cpu(), val_preds.cpu())
    
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1}/{n_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}")
    
    # Early stopping check
    if val_f1 > best_f1:
        best_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), 'best_ann_model.pt')  # best model save karo
    else:
        patience_counter += 1
        if patience_counter >= early_stop_patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

print(f"\nBest Validation F1-score: {best_f1:.4f}")

Epoch 1/30 | Train Loss: 1.1578 | Val Loss: 1.1085 | Val F1: 0.2520
Epoch 2/30 | Train Loss: 1.1165 | Val Loss: 1.1066 | Val F1: 0.2518
Epoch 3/30 | Train Loss: 1.1082 | Val Loss: 1.1019 | Val F1: 0.2574
Epoch 4/30 | Train Loss: 1.1033 | Val Loss: 1.1016 | Val F1: 0.2492
Epoch 5/30 | Train Loss: 1.0991 | Val Loss: 1.1024 | Val F1: 0.2505
Epoch 6/30 | Train Loss: 1.0948 | Val Loss: 1.1024 | Val F1: 0.2516
Epoch 7/30 | Train Loss: 1.0911 | Val Loss: 1.1011 | Val F1: 0.2559
Epoch 8/30 | Train Loss: 1.0869 | Val Loss: 1.1014 | Val F1: 0.2516
Early stopping at epoch 8

Best Validation F1-score: 0.2574


In [26]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

model.eval()
with torch.no_grad():
    val_outputs = model(X_val_t)
    val_probs = torch.sigmoid(val_outputs).cpu().numpy()
    val_preds = (val_probs >= 0.5).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_val_t.cpu(), val_preds))
print("\nClassification Report:")
print(classification_report(y_val_t.cpu(), val_preds))
print(f"\nROC-AUC: {roc_auc_score(y_val_t.cpu(), val_probs):.4f}")

Confusion Matrix:
[[37300 19238]
 [ 1482  3483]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.96      0.66      0.78     56538
         1.0       0.15      0.70      0.25      4965

    accuracy                           0.66     61503
   macro avg       0.56      0.68      0.52     61503
weighted avg       0.90      0.66      0.74     61503


ROC-AUC: 0.7422


In [27]:
from sklearn.preprocessing import LabelEncoder

# The df_model is the same as it was before one-hot encoding. (72 columns, target included)
categorical_cols = df_model.select_dtypes(include='object').columns.tolist()
numerical_cols = df_model.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_cols.remove('TARGET')

print(f"Categorical: {len(categorical_cols)}, Numerical: {len(numerical_cols)}")

# Label encode categorical columns (integer index per category, required for embeddings)
label_encoders = {}
cat_cardinalities = []  # How many unique categories are there in each categorical column

df_tab = df_model.copy()
for col in categorical_cols:
    le = LabelEncoder()
    df_tab[col] = le.fit_transform(df_tab[col])
    label_encoders[col] = le
    cat_cardinalities.append(len(le.classes_))

print(f"Cardinalities: {cat_cardinalities}")

Categorical: 12, Numerical: 59
Cardinalities: [2, 3, 2, 2, 7, 8, 5, 6, 6, 18, 7, 58]


In [28]:
from sklearn.model_selection import train_test_split

X_cat = df_tab[categorical_cols].values
X_num = df_tab[numerical_cols].values
y_tab = df_tab['TARGET'].values

# Scale the numerical features (not the categorical ones—embeddings will handle those).
scaler_tab = StandardScaler()
X_num_scaled = scaler_tab.fit_transform(X_num)

X_cat_train, X_cat_val, X_num_train, X_num_val, y_train_tab, y_val_tab = train_test_split(
    X_cat, X_num_scaled, y_tab, test_size=0.2, stratify=y_tab, random_state=42
)
print(f"Cat train: {X_cat_train.shape}, Num train: {X_num_train.shape}")

Cat train: (246008, 12), Num train: (246008, 59)


In [32]:
class TabTransformer(nn.Module):
    def __init__(self, cat_cardinalities, num_numerical, embed_dim=16, n_heads=4, n_layers=2):
        super(TabTransformer, self).__init__()
        
        # Separate embedding layer for each categorical column
        self.embeddings = nn.ModuleList([
            nn.Embedding(card, embed_dim) for card in cat_cardinalities
        ])
        
        # Self-attention layers (on top of categorical embeddings)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads, dim_feedforward=embed_dim*2, 
            dropout=0.2, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        # Normalization for numerical features
        self.num_norm = nn.BatchNorm1d(num_numerical)
        
        # Final MLP head (by combining attention output + numerical features)
        n_cat = len(cat_cardinalities)
        mlp_input_size = (n_cat * embed_dim) + num_numerical
        self.mlp = nn.Sequential(
            nn.Linear(mlp_input_size, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    
    def forward(self, x_cat, x_num):
        # Embed each categorical column, then stack them (batch, n_cat, embed_dim).
        embedded = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        embedded = torch.stack(embedded, dim=1)
        
        # Apply self-attention.
        attended = self.transformer(embedded)
        attended_flat = attended.flatten(start_dim=1)  # (batch, n_cat * embed_dim)
        
        # Numerical features by normalizing
        num_normed = self.num_norm(x_num)
        
        # Combine and pass through MLP.
        combined = torch.cat([attended_flat, num_normed], dim=1)
        return self.mlp(combined)

tab_model = TabTransformer(cat_cardinalities, len(numerical_cols)).to(device)
print(tab_model)

TabTransformer(
  (embeddings): ModuleList(
    (0): Embedding(2, 16)
    (1): Embedding(3, 16)
    (2-3): 2 x Embedding(2, 16)
    (4): Embedding(7, 16)
    (5): Embedding(8, 16)
    (6): Embedding(5, 16)
    (7-8): 2 x Embedding(6, 16)
    (9): Embedding(18, 16)
    (10): Embedding(7, 16)
    (11): Embedding(58, 16)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=16, out_features=16, bias=True)
        )
        (linear1): Linear(in_features=16, out_features=32, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=32, out_features=16, bias=True)
        (norm1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=

In [34]:
# Convert the categorical data to LongTensor (embeddings require integer indices).
X_cat_train_t = torch.LongTensor(X_cat_train).to(device)
X_num_train_t = torch.FloatTensor(X_num_train).to(device)
y_train_tab_t = torch.FloatTensor(y_train_tab).reshape(-1, 1).to(device)

X_cat_val_t = torch.LongTensor(X_cat_val).to(device)
X_num_val_t = torch.FloatTensor(X_num_val).to(device)
y_val_tab_t = torch.FloatTensor(y_val_tab).reshape(-1, 1).to(device)

print(f"Cat train tensor: {X_cat_train_t.shape}, dtype: {X_cat_train_t.dtype}")

Cat train tensor: torch.Size([246008, 12]), dtype: torch.int64


In [37]:
criterion_tab = nn.BCEWithLogitsLoss(pos_weight=pos_weight)  # same pos_weight that was calculated earlier.
optimizer_tab = optim.Adam(tab_model.parameters(), lr=1e-3)
scheduler_tab = optim.lr_scheduler.ReduceLROnPlateau(optimizer_tab, mode='min', factor=0.5, patience=3)

In [38]:
from torch.utils.data import DataLoader, TensorDataset

train_dataset_tab = TensorDataset(X_cat_train_t, X_num_train_t, y_train_tab_t)
train_loader_tab = DataLoader(train_dataset_tab, batch_size=512, shuffle=True)

n_epochs = 30
best_f1_tab = 0
patience_counter = 0
early_stop_patience = 5

for epoch in range(n_epochs):
    tab_model.train()
    total_loss = 0
    for batch_cat, batch_num, batch_y in train_loader_tab:
        optimizer_tab.zero_grad()
        outputs = tab_model(batch_cat, batch_num)
        loss = criterion_tab(outputs, batch_y)
        loss.backward()
        optimizer_tab.step()
        total_loss += loss.item()
    
    avg_train_loss = total_loss / len(train_loader_tab)
    
    # Validation
    tab_model.eval()
    with torch.no_grad():
        val_outputs = tab_model(X_cat_val_t, X_num_val_t)
        val_loss = criterion_tab(val_outputs, y_val_tab_t).item()
        val_preds = (torch.sigmoid(val_outputs) >= 0.5).float()
        val_f1 = f1_score(y_val_tab_t.cpu(), val_preds.cpu())
    
    scheduler_tab.step(val_loss)
    
    print(f"Epoch {epoch+1}/{n_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}")
    
    if val_f1 > best_f1_tab:
        best_f1_tab = val_f1
        patience_counter = 0
        torch.save(tab_model.state_dict(), 'best_tabtransformer_model.pt')
    else:
        patience_counter += 1
        if patience_counter >= early_stop_patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

print(f"\nBest Validation F1-score (TabTransformer): {best_f1_tab:.4f}")

Epoch 1/30 | Train Loss: 1.1375 | Val Loss: 1.2508 | Val F1: 0.2494
Epoch 2/30 | Train Loss: 1.1150 | Val Loss: 1.5480 | Val F1: 0.2504
Epoch 3/30 | Train Loss: 1.1111 | Val Loss: 1.5018 | Val F1: 0.2513
Epoch 4/30 | Train Loss: 1.1082 | Val Loss: 1.1132 | Val F1: 0.2562
Epoch 5/30 | Train Loss: 1.1062 | Val Loss: 1.1145 | Val F1: 0.2584
Epoch 6/30 | Train Loss: 1.1041 | Val Loss: 1.1029 | Val F1: 0.2571
Epoch 7/30 | Train Loss: 1.1030 | Val Loss: 1.8057 | Val F1: 0.2524
Epoch 8/30 | Train Loss: 1.1011 | Val Loss: 1.1400 | Val F1: 0.2563
Epoch 9/30 | Train Loss: 1.0984 | Val Loss: 1.1005 | Val F1: 0.2590
Epoch 10/30 | Train Loss: 1.0961 | Val Loss: 1.1005 | Val F1: 0.2593
Epoch 11/30 | Train Loss: 1.0954 | Val Loss: 2.0365 | Val F1: 0.2582
Epoch 12/30 | Train Loss: 1.0927 | Val Loss: 2.8984 | Val F1: 0.2554
Epoch 13/30 | Train Loss: 1.0917 | Val Loss: 2.1314 | Val F1: 0.2619
Epoch 14/30 | Train Loss: 1.0846 | Val Loss: 1.9412 | Val F1: 0.2538
Epoch 15/30 | Train Loss: 1.0821 | Val Loss

In [39]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

tab_model.eval()
with torch.no_grad():
    val_outputs = tab_model(X_cat_val_t, X_num_val_t)
    val_probs_tab = torch.sigmoid(val_outputs).cpu().numpy()
    val_preds_tab = (val_probs_tab >= 0.5).astype(int)

print("TabTransformer Confusion Matrix:")
print(confusion_matrix(y_val_tab_t.cpu(), val_preds_tab))
print("\nTabTransformer Classification Report:")
print(classification_report(y_val_tab_t.cpu(), val_preds_tab))
print(f"\nTabTransformer ROC-AUC: {roc_auc_score(y_val_tab_t.cpu(), val_probs_tab):.4f}")

TabTransformer Confusion Matrix:
[[37727 18811]
 [ 1487  3478]]

TabTransformer Classification Report:
              precision    recall  f1-score   support

         0.0       0.96      0.67      0.79     56538
         1.0       0.16      0.70      0.26      4965

    accuracy                           0.67     61503
   macro avg       0.56      0.68      0.52     61503
weighted avg       0.90      0.67      0.75     61503


TabTransformer ROC-AUC: 0.7462


In [40]:
!pip install shap -q
import shap

In [41]:
# Load Best model weights
model.load_state_dict(torch.load('best_ann_model.pt'))
model.eval()
print("Model loaded successfully")

Model loaded successfully


In [45]:
import numpy as np

def model_predict(x):
    x_tensor = torch.FloatTensor(x).to(device)
    with torch.no_grad():
        output = model(x_tensor)
        prob = torch.sigmoid(output).cpu().numpy()
    return prob

# SHAP needs a background dataset (take a small sample from the training data for speed).
background = X_train[np.random.choice(X_train.shape[0], 100, replace=False)]

explainer = shap.KernelExplainer(model_predict, background)

In [46]:
# Take only 20 validation samples (for speed)—in real inference, this will be a per-request operation, one at a time.
sample_idx = np.random.choice(X_val.shape[0], 20, replace=False)
X_sample = X_val[sample_idx]

shap_values = explainer.shap_values(X_sample, nsamples=100)

print(f"SHAP values shape: {np.array(shap_values).shape}")

  0%|          | 0/20 [00:00<?, ?it/s]

SHAP values shape: (20, 171, 1)


In [48]:
feature_names = X.columns.tolist()  # one-hot encoded columns name

# firstly sample applicant explanation 
applicant_shap = shap_values[0].flatten()  # shap values for 1st sample
top_features_idx = np.argsort(np.abs(applicant_shap))[-5:][::-1]  # top 5 by absolute impact

print("Top 5 factors for this applicant's decision:")
for idx in top_features_idx:
    direction = "increases risk" if applicant_shap[idx] > 0 else "decreases risk"
    print(f"  {feature_names[idx]}: SHAP value = {applicant_shap[idx]:.4f} ({direction})")

Top 5 factors for this applicant's decision:
  EXT_SOURCE_2: SHAP value = -0.0621 (decreases risk)
  FLAG_PHONE: SHAP value = -0.0343 (decreases risk)
  NAME_FAMILY_STATUS_Married: SHAP value = -0.0299 (decreases risk)
  AMT_ANNUITY: SHAP value = 0.0271 (increases risk)
  LIVE_REGION_NOT_WORK_REGION: SHAP value = 0.0208 (increases risk)


In [49]:
!pip install dice-ml -q
import dice_ml
from dice_ml import Dice

# Data prep for DiCE
df_dice = df_model.copy()
d = dice_ml.Data(dataframe=df_dice, continuous_features=numerical_cols, outcome_name='TARGET')

# Wrap PyTorch model for DiCE (sklearn-style wrapper)
class ModelWrapper:
    def __init__(self, model, feature_names):
        self.model = model
        self.feature_names = feature_names
    def predict_proba(self, X):
        X_enc = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
        X_enc = X_enc.reindex(columns=feature_names, fill_value=0)
        X_scaled = scaler.transform(X_enc)
        probs = model_predict(X_scaled)
        return np.hstack([1-probs, probs])

wrapped_model = ModelWrapper(model, feature_names)
m = dice_ml.Model(model=wrapped_model, backend='sklearn', model_type='classifier')
exp = Dice(d, m)

# Example: rejected applicant ke liye counterfactual generate karo
query_instance = df_dice[df_dice['TARGET']==1].drop(columns=['TARGET']).iloc[[0]]
cf = exp.generate_counterfactuals(query_instance, total_CFs=3, desired_class="opposite")
cf.visualize_as_dataframe()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 7.9 MB/s eta 0:00:0000:0100:01


100%|██████████| 1/1 [00:02<00:00,  2.59s/it]

Query instance (original outcome : 1)


,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,...,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,TARGET
0,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1



Diverse Counterfactual set (new outcome: 0)


,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,...,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,TARGET
0,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1
1,Cash loans,M,N,Y,6,202500.0,406597.5,24700.5,351000.0,Unaccompanied,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,0
2,Cash loans,M,Y,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,...,0,0,0,0.0,0.0,0.0,0.0,158.2,1.0,0


In [51]:
def mc_dropout_predict(model, X, n_passes=50):
    model.train()  # dropout ON rakhna hai
    preds = []
    X_t = torch.FloatTensor(X).to(device)
    with torch.no_grad():
        for _ in range(n_passes):
            out = torch.sigmoid(model(X_t)).cpu().numpy()
            preds.append(out)
    preds = np.array(preds)
    mean_pred = preds.mean(axis=0)
    uncertainty = preds.std(axis=0)
    model.eval()
    return mean_pred, uncertainty

mean_probs, uncertainty = mc_dropout_predict(model, X_val[:100])
print(f"Sample mean probs: {mean_probs[:5].flatten()}")
print(f"Sample uncertainty: {uncertainty[:5].flatten()}")

Sample mean probs: [0.40689173 0.36680382 0.6843493  0.59969574 0.59668165]
Sample uncertainty: [0.03854854 0.03491525 0.04303451 0.04530161 0.05383839]


In [52]:
from sklearn.linear_model import LogisticRegression

# Poore validation set pe MC Dropout chalao (calibration fit karne ke liye)
raw_probs_val, _ = mc_dropout_predict(model, X_val, n_passes=20)

platt = LogisticRegression()
platt.fit(raw_probs_val.reshape(-1, 1), y_val)

calibrated_probs = platt.predict_proba(raw_probs_val.reshape(-1, 1))[:, 1]
print("Calibration fitted successfully")
print("Sample calibrated probs:", calibrated_probs[:5])

Calibration fitted successfully
Sample calibrated probs: [0.04657767 0.03921128 0.18970263 0.15230229 0.12353078]
